Starting Reinforced Learning

In [7]:
# code from AI to learn from
import gymnasium as gym
import numpy as np
import random
import time

# 1. Create the 2D game environment using Gymnasium
# "is_slippery=False" makes it easier for the AI to learn at first
env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")

# 2. Setup the AI's memory (The Q-Table Spreadsheet)
# Rows = 16 grid squares, Columns = 4 directions (Left, Down, Right, Up)
state_size = env.observation_space.n
action_size = env.action_space.n
q_table = np.zeros((state_size, action_size))

# Learning settings (Hyperparameters)
learning_rate = 0.8
discount_factor = 0.95
exploration_rate = 1.0  # Starts out 100% random to explore the map
exploration_decay = 0.995


# 3. Dynamic Training Loop
for episode in range(1000):
    state, info = env.reset()
    done = False
    
    while not done:
        # Exploration vs Exploitation choice
        # If a random number is less than exploration_rate, guess randomly.
        # Otherwise, look at the spreadsheet and pick the best known move!
        if random.uniform(0, 1) < exploration_rate:
            action = env.action_space.sample() # Random guess
        else:
            action = np.argmax(q_table[state, :]) # Smart choice from spreadsheet
            
        # Take the action in the Gymnasium 2D game
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # THE DYNAMIC REINFORCEMENT MATH:
        # Update the spreadsheet cell based on the reward received
        best_future_q = np.max(q_table[next_state, :])
        q_table[state, action] = q_table[state, action] + learning_rate * (reward + discount_factor * best_future_q - q_table[state, action])
        
        state = next_state
        
    # Slowly stop guessing randomly as the spreadsheet gets smarter
    exploration_rate *= exploration_decay

print("✅ Training Complete!")


✅ Training Complete!


In [8]:
# We build a fresh environment with "human" mode to open the Pygame window
visual_env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")

state, info = visual_env.reset()
done = False

# The AI plays 1 round using its completed spreadsheet memory
while not done:
    # Always pick the absolute best move from the learned Q-table
    action = np.argmax(q_table[state, :]) 
    
    state, reward, terminated, truncated, info = visual_env.step(action)
    done = terminated or truncated
    
    # Pause for 0.5 seconds per step so human eyes can keep up with the window!
    time.sleep(0.5) 
visual_env.close() # Closes the popup window safely

# My breakdown

**Part 1: Training**
